# Лабораторная работа №1

## Компьютерная лексикография, терминологические базы (TBX) и Translation Memory (TMX) в Smartcat

**ФИО студента:** Карпов Семён Алексеевич
**Группа:** ИНФА-221
**Вариант:** 5 (предметная область: Юриспруденция: уголовное, гражданское и процессуальное право)

---

**Цель:** научиться программно обрабатывать стандарты обмена терминологическими
данными (TBX) и памятью переводов (TMX), реализовать поиск нечётких совпадений
(Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в
CAT-систему Smartcat.

**Порядок выполнения:**

1. Выполните ячейки Блока 1 (установка библиотек) и Блока 2 (данные).
2. Реализуйте четыре функции в Блоке 4 вместо маркеров `# TODO`.
3. Запустите Блок 5 — все `assert` должны пройти без ошибок.
4. Соберите файлы своего варианта, экспортируйте CSV, оформите отчёт.

**Оценивание (10 баллов):** парсинг TBX — 2, парсинг TMX — 2, Fuzzy Match — 3,
интеграция глоссария — 2, оформление — 1.

## Блок 1. Setup — установка библиотек

In [ ]:
!pip install -q Levenshtein pandas lxml

In [ ]:
from __future__ import annotations

import os
import re
import unicodedata
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    from lxml import etree as ET

    LXML_AVAILABLE = True
except ImportError:
    import xml.etree.ElementTree as ET

    LXML_AVAILABLE = False

try:
    import Levenshtein as _lev

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна между двумя строками."""
        return _lev.distance(a, b)

except ImportError:

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (резервная реализация на чистом Python)."""
        if a == b:
            return 0
        if not a:
            return len(b)
        if not b:
            return len(a)
        previous = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(min(previous[j] + 1,
                                   current[j - 1] + 1,
                                   previous[j - 1] + (char_a != char_b)))
            previous = current
        return previous[-1]


# ВАЖНО: атрибут xml:lang в парсере раскрывается в полное имя с пространством имён
XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"

pd.set_option("display.max_colwidth", 60)
print("Окружение готово. XML backend:", "lxml" if LXML_AVAILABLE else "ElementTree")

Окружение готово. XML backend: lxml


## Блок 2. Теория: структура стандартов

### 2.1. TBX (TermBase eXchange, ISO 30042)

Формат обмена терминологическими базами. Организован **по понятиям**
(concept-oriented): одна `termEntry` = одно понятие, внутри — по одной `langSet`
на каждый язык.

```xml
<martif type="TBX" xml:lang="en">
  <martifHeader>…</martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Предметная область</descrip>
      <langSet xml:lang="en">
        <tig>
          <term>flight controller</term>
          <termNote type="partOfSpeech">noun</termNote>
        </tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp>
          <term>полётный контроллер</term>
        </termGrp></ntig>
      </langSet>
    </termEntry>
  </body></text>
</martif>
```

| Тег | Назначение |
|-----|------------|
| `martif` | корень документа TBX-Core v2 |
| `termEntry` | терминологическая статья (одно понятие) |
| `langSet` | языковая секция, язык задаётся атрибутом `xml:lang` |
| `tig` / `ntig` | компактная / расширенная группа термина |
| `term` | сам термин |
| `termNote` | грамматическая или иная помета (`partOfSpeech`, `usage`) |
| `descrip` | описание: `definition`, `subjectField`, `context` |

### 2.2. TMX (Translation Memory eXchange, версия 1.4b)

Формат обмена памятью переводов. Организован **по сегментам**.

```xml
<tmx version="1.4">
  <header srclang="en" segtype="sentence" datatype="plaintext"
          creationtool="…" adminlang="en" o-tmf="TMX"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Aerospace</prop>
      <tuv xml:lang="en"><seg>The autopilot holds the altitude.</seg></tuv>
      <tuv xml:lang="ru"><seg>Автопилот выдерживает высоту.</seg></tuv>
    </tu>
  </body>
</tmx>
```

| Тег | Назначение |
|-----|------------|
| `header` | метаданные памяти: инструмент, язык оригинала, тип сегментации |
| `tu` | translation unit — единица перевода |
| `tuv` | translation unit variant — вариант на одном языке (`xml:lang`) |
| `seg` | текст сегмента |
| `prop` | произвольное свойство (`domain`, `client`, `project`) |

### 2.3. Типы совпадений в CAT-системах

| Совпадение | Диапазон | Действие переводчика |
|------------|----------|----------------------|
| ICE (101 %) | 100 % + совпал контекст | подстановка без правки |
| Exact | 100 % | проверить контекст |
| High Fuzzy | 95–99 % | косметическая правка |
| Fuzzy | 75–94 % | редактирование |
| No Match | < 75 % | перевод с нуля |

## Блок 3. Данные

Ячейка ниже создаёт **демонстрационные** файлы `demo.tbx` и `demo.tmx` — они
нужны только для автотестов из Блока 5. Файлы своего варианта
(`variant_NN.tbx` — не менее 15 терминов, `variant_NN.tmx` — не менее 10
предложений) вы создаёте самостоятельно: либо аналогичным кодом, либо выгрузкой
из Smartcat, либо вручную.

In [ ]:
DEMO_TBX = """<?xml version="1.0" encoding="UTF-8"?>
<martif type="TBX" xml:lang="en">
  <martifHeader><fileDesc><titleStmt><title>Demo glossary</title></titleStmt>
  </fileDesc></martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en">
        <tig><term>flight controller</term>
        <termNote type="partOfSpeech">noun</termNote>
        <descrip type="definition">Onboard flight computer.</descrip></tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp><term>полётный контроллер</term></termGrp></ntig>
      </langSet>
    </termEntry>
    <termEntry id="tid-002">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>payload</term>
        <termNote type="partOfSpeech">noun</termNote></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>целевая нагрузка</term>
        </termGrp></ntig></langSet>
    </termEntry>
    <termEntry id="tid-003">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>telemetry link</term></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>канал телеметрии</term>
        </termGrp></ntig></langSet>
    </termEntry>
  </body></text>
</martif>
"""

DEMO_TMX = """<?xml version="1.0" encoding="UTF-8"?>
<tmx version="1.4">
  <header creationtool="demo" creationtoolversion="1.0" segtype="sentence"
          o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The flight controller stabilises the aircraft.</seg></tuv>
      <tuv xml:lang="ru"><seg>Полётный контроллер стабилизирует аппарат.</seg></tuv>
    </tu>
    <tu tuid="2">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The operator checks the telemetry link.</seg></tuv>
      <tuv xml:lang="ru"><seg>Оператор проверяет канал телеметрии.</seg></tuv>
    </tu>
    <tu tuid="3">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The payload is mounted under the fuselage.</seg></tuv>
      <tuv xml:lang="ru"><seg>Целевая нагрузка устанавливается под фюзеляжем.</seg></tuv>
    </tu>
  </body>
</tmx>
"""

with open("demo.tbx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TBX)
with open("demo.tmx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TMX)

print("Созданы файлы:", [f for f in os.listdir(".") if f.endswith((".tbx", ".tmx"))])

Созданы файлы: ['variant_6.tbx', 'demo.tmx', 'demo.tbx', 'variant_5.tbx', 'variant_5.tmx', 'variant_6.tmx']


## Блок 4. Задание — реализуйте четыре функции

Требования ко всем функциям: типизация из `typing`, соответствие PEP 8,
обработка отсутствующих элементов без падения (`None`-проверки).

**Подсказки:**

* язык секции читается как `element.get(XML_LANG)`, а не `element.get("lang")`;
* обход `<term>` внутри `langSet` удобно делать через `lang_set.iter("term")` —
  это покрывает и `tig`, и `ntig/termGrp`;
* текст сегмента лучше собирать через `"".join(seg.itertext())` — так не
  теряется текст при наличии инлайновых тегов;
* при подстановке терминов сортируйте их по убыванию длины.

In [ ]:
def parse_tbx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TBX-глоссарий в плоскую таблицу терминологических пар."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    root = ET.parse(file_path).getroot()
    rows: List[Dict[str, str]] = []

    for entry in root.iter("termEntry"):
        subject = ""
        subject_node = entry.find("descrip[@type='subjectField']")
        if subject_node is not None and subject_node.text:
            subject = subject_node.text.strip()

        by_lang: Dict[str, Dict[str, str]] = {}

        for lang_set in entry.findall("langSet"):
            lang = (lang_set.get(XML_LANG) or "").split("-")[0].lower()
            terms = [t.text.strip() for t in lang_set.iter("term") if (t.text or "").strip()]
            if not terms:
                continue

            note = lang_set.find(".//termNote[@type='partOfSpeech']")
            definition = lang_set.find(".//descrip[@type='definition']")

            by_lang[lang] = {
                "term": terms[0],
                "pos": (note.text or "").strip() if note is not None else "",
                "definition": (definition.text or "").strip() if definition is not None else "",
            }

        if src_lang in by_lang and tgt_lang in by_lang:
            rows.append({
                "entry_id": entry.get("id", ""),
                "source_term": by_lang[src_lang]["term"],
                "target_term": by_lang[tgt_lang]["term"],
                "pos": by_lang[src_lang]["pos"] or by_lang[tgt_lang]["pos"],
                "definition": by_lang[src_lang]["definition"],
                "subject_field": subject,
            })

    return pd.DataFrame(
        rows,
        columns=["entry_id", "source_term", "target_term", "pos", "definition", "subject_field"],
    )


def parse_tmx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TMX-память переводов в таблицу параллельных сегментов."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    def _seg_text(tuv) -> str:
        seg = tuv.find("seg")
        if seg is None:
            return ""
        return re.sub(r"\s+", " ", "".join(seg.itertext())).strip()

    root = ET.parse(file_path).getroot()
    rows: List[Dict[str, Any]] = []

    for order, tu in enumerate(root.iter("tu"), start=1):
        segments: Dict[str, str] = {}

        for tuv in tu.findall("tuv"):
            lang = (tuv.get(XML_LANG) or tuv.get("lang") or "").split("-")[0].lower()
            text = _seg_text(tuv)
            if lang and text:
                segments[lang] = text

        if src_lang not in segments or tgt_lang not in segments:
            continue

        domain_node = tu.find("prop[@type='domain']")
        rows.append({
            "tuid": tu.get("tuid", str(order)),
            "source_segment": segments[src_lang],
            "target_segment": segments[tgt_lang],
            "domain": (domain_node.text or "").strip() if domain_node is not None else "",
            "src_len": len(segments[src_lang]),
        })

    frame = pd.DataFrame(
        rows,
        columns=["tuid", "source_segment", "target_segment", "domain", "src_len"],
    )
    frame["prev_source"] = frame["source_segment"].shift(1).fillna("")
    return frame


def calculate_fuzzy_match(source_query: str, tm_df: pd.DataFrame,
                          threshold: float = 70.0,
                          context_before: Optional[str] = None,
                          top_n: int = 5) -> List[Dict[str, Any]]:
    """Найти в памяти переводов сегменты, похожие на запрос."""
    if tm_df.empty:
        return []

    results: List[Dict[str, Any]] = []

    for _, row in tm_df.iterrows():
        norm_query = normalize(source_query)
        norm_segment = normalize(str(row["source_segment"]))

        lev = levenshtein_ratio(norm_query, norm_segment)
        tsr = token_sort_ratio(source_query, str(row["source_segment"]))
        score = max(lev, tsr)

        if score < threshold:
            continue

        context_matched = bool(
            context_before is not None
            and normalize(str(context_before)) == normalize(str(row.get("prev_source", "")))
        )

        results.append({
            "tuid": row["tuid"],
            "source_segment": row["source_segment"],
            "target_segment": row["target_segment"],
            "score": score,
            "lev_ratio": lev,
            "token_sort_ratio": tsr,
            "match_type": classify_match(score, context_matched),
        })

    results.sort(key=lambda item: item["score"], reverse=True)
    return results[:top_n]


def apply_term_glossary(text: str, glossary_df: pd.DataFrame,
                        template: str = "{source} [{target}]") -> str:
    """Разметить исходный текст переводными эквивалентами из глоссария."""
    if glossary_df.empty or not text:
        return text

    pairs = sorted(
        zip(glossary_df["source_term"], glossary_df["target_term"]),
        key=lambda pair: len(str(pair[0])),
        reverse=True,
    )

    protected: List[str] = []

    def _store(replacement: str) -> str:
        protected.append(replacement)
        return f"\x00{len(protected) - 1}\x00"

    result = text

    for source_term, target_term in pairs:
        if not source_term:
            continue

        pattern = re.compile(
            rf"(?<!\w){re.escape(str(source_term))}(?!\w)",
            re.IGNORECASE,
        )
        result = pattern.sub(
            lambda match: _store(
                template.format(source=match.group(0), target=target_term)
            ),
            result,
        )

    return re.sub(r"\x00(\d+)\x00", lambda m: protected[int(m.group(1))], result)

### 4.1. Вспомогательные функции (реализуйте при необходимости)

Эти функции не оцениваются отдельно, но нужны для `calculate_fuzzy_match`.

In [ ]:
def normalize(text: str) -> str:
    """Нормализовать строку: NFKC, нижний регистр, без пунктуации, одиночные пробелы."""
    text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def levenshtein_ratio(a: str, b: str) -> float:
    """Процент сходства строк на основе расстояния Левенштейна."""
    if not a and not b:
        return 100.0
    longest = max(len(a), len(b))
    return round((1.0 - edit_distance(a, b) / longest) * 100.0, 2)


def token_sort_ratio(a: str, b: str) -> float:
    """Процент сходства при игнорировании порядка слов."""
    sorted_a = " ".join(sorted(normalize(a).split()))
    sorted_b = " ".join(sorted(normalize(b).split()))
    return levenshtein_ratio(sorted_a, sorted_b)


def classify_match(score: float, context_matched: bool) -> str:
    """Определить тип совпадения по проценту и признаку совпадения контекста."""
    if score >= 100.0:
        return "ICE (101%)" if context_matched else "Exact (100%)"
    if score >= 95.0:
        return "High Fuzzy"
    if score >= 75.0:
        return "Fuzzy"
    return "Low Fuzzy"

## Блок 5. Автотесты

Запустите ячейки ниже **после** реализации функций. Все проверки должны
завершиться без `AssertionError`. Тесты работают на демо-файлах из Блока 3.

In [ ]:
# --- Тест 1: метрики сходства -------------------------------------------
assert abs(levenshtein_ratio("telemetry", "telemetry") - 100.0) < 1e-6, \
    "Одинаковые строки должны давать 100 %"
assert levenshtein_ratio("", "") == 100.0, "Две пустые строки — 100 %"
assert 0.0 <= levenshtein_ratio("payload", "airframe") < 50.0, \
    "Непохожие строки должны давать низкий процент"
assert abs(token_sort_ratio("angle of attack", "attack of angle") - 100.0) < 1e-6, \
    "Token Sort Ratio должен игнорировать порядок слов"
assert normalize("  Angle   OF ATTACK, critical! ") == "angle of attack critical", \
    "normalize: нижний регистр, без пунктуации, одиночные пробелы"
print("Тест 1 (метрики сходства) — OK")

Тест 1 (метрики сходства) — OK


In [ ]:
# --- Тест 2: парсинг TBX -------------------------------------------------
glossary_df = parse_tbx("demo.tbx")

assert isinstance(glossary_df, pd.DataFrame), "parse_tbx должен вернуть DataFrame"
assert len(glossary_df) == 3, f"Ожидается 3 термина, получено {len(glossary_df)}"
for column in ["entry_id", "source_term", "target_term", "pos",
               "definition", "subject_field"]:
    assert column in glossary_df.columns, f"Нет колонки '{column}'"

row = glossary_df.loc[glossary_df["source_term"] == "flight controller"].iloc[0]
assert row["target_term"] == "полётный контроллер", "Неверный русский эквивалент"
assert row["entry_id"] == "tid-001", "Не извлечён атрибут id у termEntry"
assert row["pos"] == "noun", "Не извлечён termNote[@type='partOfSpeech']"
assert "telemetry link" in set(glossary_df["source_term"]), \
    "Термин из <tig> без termNote должен быть извлечён"
print("Тест 2 (парсинг TBX) — OK")
glossary_df.head()

Тест 2 (парсинг TBX) — OK


,entry_id,source_term,target_term,pos,definition,subject_field
0,tid-001,flight controller,полётный контроллер,noun,Onboard flight computer.,Demo
1,tid-002,payload,целевая нагрузка,noun,,Demo
2,tid-003,telemetry link,канал телеметрии,,,Demo


In [ ]:
# --- Тест 3: парсинг TMX -------------------------------------------------
tm_df = parse_tmx("demo.tmx")

assert isinstance(tm_df, pd.DataFrame), "parse_tmx должен вернуть DataFrame"
assert len(tm_df) == 3, f"Ожидается 3 единицы перевода, получено {len(tm_df)}"
for column in ["tuid", "source_segment", "target_segment", "domain",
               "src_len", "prev_source"]:
    assert column in tm_df.columns, f"Нет колонки '{column}'"

first = tm_df.iloc[0]
assert first["source_segment"].startswith("The flight controller"), \
    "Не извлечён текст <seg> исходного языка"
assert first["target_segment"].startswith("Полётный контроллер"), \
    "Не извлечён текст <seg> языка перевода"
assert first["prev_source"] == "", "У первой строки prev_source должен быть пустым"
assert tm_df.iloc[1]["prev_source"] == first["source_segment"], \
    "prev_source должен содержать предыдущий исходный сегмент"
assert first["src_len"] == len(first["source_segment"]), "Неверная длина сегмента"
print("Тест 3 (парсинг TMX) — OK")
tm_df.head()

Тест 3 (парсинг TMX) — OK


,tuid,source_segment,target_segment,domain,src_len,prev_source
0,1,The flight controller stabilises the aircraft.,Полётный контроллер стабилизирует аппарат.,Demo,46,
1,2,The operator checks the telemetry link.,Оператор проверяет канал телеметрии.,Demo,39,The flight controller stabilises the aircraft.
2,3,The payload is mounted under the fuselage.,Целевая нагрузка устанавливается под фюзеляжем.,Demo,42,The operator checks the telemetry link.


In [ ]:
# --- Тест 4: Fuzzy Match и ICE ------------------------------------------
exact_query = "The operator checks the telemetry link."
matches = calculate_fuzzy_match(exact_query, tm_df, threshold=70.0)
assert matches, "Точное совпадение обязано быть найдено"
assert abs(matches[0]["score"] - 100.0) < 1e-6, "Точное совпадение — 100 %"
assert matches[0]["match_type"] == "Exact (100%)", \
    "Без контекста 100 % — это Exact, а не ICE"

ice = calculate_fuzzy_match(
    exact_query, tm_df, threshold=70.0,
    context_before="The flight controller stabilises the aircraft.")
assert ice[0]["match_type"] == "ICE (101%)", \
    "При совпадении предыдущего сегмента 100 % должно стать ICE"

fuzzy = calculate_fuzzy_match("The engineer checks the telemetry link.",
                              tm_df, threshold=70.0)
assert fuzzy, "Похожий сегмент должен попадать в выдачу"
assert 75.0 <= fuzzy[0]["score"] < 100.0, \
    f"Ожидается Fuzzy 75–99 %, получено {fuzzy[0]['score']}"
assert fuzzy[0]["match_type"] in {"Fuzzy", "High Fuzzy"}, "Неверный тип совпадения"

scores = [m["score"] for m in calculate_fuzzy_match("telemetry link check",
                                                    tm_df, threshold=0.0)]
assert scores == sorted(scores, reverse=True), \
    "Результаты должны быть отсортированы по убыванию сходства"
assert calculate_fuzzy_match("Совершенно посторонний текст про бухгалтерию.",
                             tm_df, threshold=75.0) == [], \
    "Ниже порога результатов быть не должно"
assert len(calculate_fuzzy_match("telemetry", tm_df, threshold=0.0, top_n=2)) <= 2, \
    "Параметр top_n не учитывается"
print("Тест 4 (Fuzzy Match и ICE) — OK")

Тест 4 (Fuzzy Match и ICE) — OK


In [ ]:
# --- Тест 5: подстановка терминов ---------------------------------------
marked = apply_term_glossary("The flight controller reads the payload status.",
                             glossary_df)
assert "[полётный контроллер]" in marked, "Термин из глоссария не подставлен"
assert "[целевая нагрузка]" in marked, "Подставлен не каждый найденный термин"
assert "flight controller" in marked, "Исходный термин должен сохраняться"

assert apply_term_glossary("No terms here at all.", glossary_df) == \
    "No terms here at all.", "Текст без терминов не должен меняться"

case_test = apply_term_glossary("Payload mass is 2 kg.", glossary_df)
assert "Payload [целевая нагрузка]" in case_test, \
    "Поиск должен быть регистронезависимым, а регистр текста — сохраняться"

assert "[целевая нагрузка]" not in apply_term_glossary("payloads are heavy",
                                                       glossary_df), \
    "Совпадение должно быть по границам слов, а не по подстроке"
print("Тест 5 (подстановка терминов) — OK")
print(marked)

Тест 5 (подстановка терминов) — OK
The flight controller [полётный контроллер] reads the payload [целевая нагрузка] status.


## Блок 6. Работа со своим вариантом

После прохождения всех тестов выполните задание на данных своего варианта.

In [ ]:
VARIANT = 5
SURNAME = "Karpov"

TBX_FILE = f"variant_{VARIANT}.tbx"
TMX_FILE = f"variant_{VARIANT}.tmx"

SUBJECT_FIELD = "Юриспруденция: договорное и контрактное право"

TERMS_V5: List[Dict[str, str]] = [
    {
    "en": "abate",
    "ru": "положить конец; прекратить",
    "pos": "verb",
    "def_en": "To put an end to, nullify, or reduce in amount, especially proportionately."
  },
  {
    "en": "abduction",
    "ru": "преступление, заключающееся в уводе жены, ребенка или подопечного",
    "pos": "noun",
    "def_en": "The criminal taking or capture of an individual against their will or without their consent, generally by means of persuasion, fraud, or force."
  },
  {
    "en": "abide by",
    "ru": "действовать в соответствии (с чем-либо)",
    "pos": "phrase",
    "def_en": "To accept, comply with, or act in accordance with a law, agreement, or decision."
  },
  {
    "en": "abrogate",
    "ru": "прекратить действие (закона); упразднять",
    "pos": "verb",
    "def_en": "To formally annul, repeal, or abolish a law, custom, or contract through authoritative action."
  },
  {
    "en": "abscond",
    "ru": "скрываться; бежать из-под стражи",
    "pos": "verb",
    "def_en": "To leave a jurisdiction secretly or suddenly, especially to avoid arrest, prosecution, or legal process."
  },
  {
    "en": "accessory",
    "ru": "пособник; соучастник",
    "pos": "noun",
    "def_en": "A person who, though not present at the commission of a crime, contributes to or assists in its commission."
  },
  {
    "en": "accomplice",
    "ru": "соучастник; пособник",
    "pos": "noun",
    "def_en": "A person who knowingly, voluntarily, or intentionally assists another in the commission of a crime."
  },
  {
    "en": "acquittal",
    "ru": "оправдание (по суду)",
    "pos": "noun",
    "def_en": "A legal and formal certification of the innocence of a person charged with a crime; a finding of not guilty."
  },
  {
    "en": "actus reus",
    "ru": "виновное действие",
    "pos": "noun",
    "def_en": "The external, physical elements of a crime, including a voluntary act, omission, or state of affairs, as required by statute."
  },
  {
    "en": "admissible evidence",
    "ru": "допустимое доказательство",
    "pos": "phrase",
    "def_en": "Evidence that is relevant, material, and not excluded by law, and may therefore be presented before the trier of fact."
  },
  {
    "en": "adoption",
    "ru": "усыновление, удочерение",
    "pos": "noun",
    "def_en": "The legal process of establishing a parent-child relationship between persons not related by blood."
  },
  {
    "en": "adversary system",
    "ru": "состязательная система",
    "pos": "phrase",
    "def_en": "A legal system in which two opposing parties present their cases before a neutral court."
  },
  {
    "en": "affidavit",
    "ru": "аффидевит; нотариально заверенное заявление",
    "pos": "noun",
    "def_en": "A written statement of facts confirmed by the oath or affirmation of the party making it."
  },
  {
    "en": "affirmation",
    "ru": "подтверждение",
    "pos": "noun",
    "def_en": "A solemn declaration made under penalty of perjury, in place of an oath, by a person who conscientiously objects to taking an oath."
  },
  {
    "en": "aggravated assault",
    "ru": "нападение при отягчающих обстоятельствах",
    "pos": "phrase",
    "def_en": "An unlawful attack by one person upon another for the purpose of inflicting severe or aggravated bodily injury, often with a deadly weapon."
  },
  {
    "en": "alibi",
    "ru": "алиби",
    "pos": "noun",
    "def_en": "A defense asserting that the accused was somewhere other than the scene of the crime at the time it was committed."
  },
  {
    "en": "alimony",
    "ru": "алименты на содержание",
    "pos": "noun",
    "def_en": "Court-ordered financial support paid by one spouse to the other during or after divorce proceedings."
  },
  {
    "en": "allegation",
    "ru": "голословное утверждение",
    "pos": "noun",
    "def_en": "A claim or assertion made without proof, especially in a legal pleading."
  },
  {
    "en": "ambiguity",
    "ru": "двусмысленность (документа)",
    "pos": "noun",
    "def_en": "The state of a document or statement having more than one possible meaning."
  },
  {
    "en": "amend",
    "ru": "вносить поправку",
    "pos": "verb",
    "def_en": "To alter, modify, or change a document, especially a legal pleading or constitution."
  },
  {
    "en": "annulment",
    "ru": "признание брака недействительным",
    "pos": "noun",
    "def_en": "A legal decree declaring a marriage void, as if it never legally existed."
  },
  {
    "en": "answer",
    "ru": "возражения ответчика (по делу)",
    "pos": "noun",
    "def_en": "A formal written response by a defendant to a plaintiff's complaint, admitting or denying the allegations."
  },
  {
    "en": "appeal",
    "ru": "апелляционная жалоба; обжалование",
    "pos": "noun",
    "def_en": "A proceeding by which a party seeks review by a higher court of a lower court's decision."
  },
  {
    "en": "appellant",
    "ru": "апеллянт; истец по апелляции",
    "pos": "noun",
    "def_en": "The party who initiates an appeal to a higher court."
  },
  {
    "en": "appellee",
    "ru": "ответчик (в апелляционном суде)",
    "pos": "noun",
    "def_en": "The party against whom an appeal is taken; the responding party in an appellate proceeding."
  },
  {
    "en": "arbitration",
    "ru": "арбитражное разбирательство",
    "pos": "noun",
    "def_en": "A process in which a neutral third party (arbitrator) resolves a dispute by rendering a binding decision."
  },
  {
    "en": "arraignment",
    "ru": "предъявление обвинения в суде",
    "pos": "noun",
    "def_en": "A court proceeding in which a criminal defendant is brought before a judge to be formally charged and enter a plea."
  },
  {
    "en": "arrest warrant",
    "ru": "ордер на арест",
    "pos": "phrase",
    "def_en": "A court order directing law enforcement to arrest a specified person."
  },
  {
    "en": "arson",
    "ru": "умышленный поджог",
    "pos": "noun",
    "def_en": "The malicious and willful burning of a dwelling or other structure."
  },
  {
    "en": "assault",
    "ru": "покушение на нанесение удара; угроза насилием",
    "pos": "noun",
    "def_en": "An intentional act that creates a reasonable apprehension of imminent harmful or offensive contact."
  },
  {
    "en": "assignee",
    "ru": "цессионарий; правопреемник",
    "pos": "noun",
    "def_en": "A person to whom a right or property is transferred."
  },
  {
    "en": "assignor",
    "ru": "цедент; лицо, передающее право",
    "pos": "noun",
    "def_en": "A person who transfers a right or property to another."
  },
  {
    "en": "attachment",
    "ru": "наложение ареста на имущество",
    "pos": "noun",
    "def_en": "A legal process by which a court seizes property to satisfy a judgment or secure jurisdiction."
  },
  {
    "en": "bail",
    "ru": "залог; поручительство",
    "pos": "noun",
    "def_en": "Security given for the release of a criminal defendant from custody, ensuring their appearance at trial."
  },
  {
    "en": "bankruptcy",
    "ru": "банкротство; несостоятельность",
    "pos": "noun",
    "def_en": "A legal proceeding for a person or business that cannot repay debts to creditors."
  },
  {
    "en": "battery",
    "ru": "избиение; незаконное применение силы",
    "pos": "noun",
    "def_en": "The intentional use of force or violence upon another person without consent."
  },
  {
    "en": "bench trial",
    "ru": "суд без участия жюри",
    "pos": "phrase",
    "def_en": "A trial in which a judge, rather than a jury, decides all issues of fact and law."
  },
  {
    "en": "bench warrant",
    "ru": "ордер на арест, выданный судьей",
    "pos": "phrase",
    "def_en": "A warrant issued by a judge for the arrest of a person who has failed to appear in court or comply with a court order."
  },
  {
    "en": "beneficiary",
    "ru": "бенефициар; выгодоприобретатель",
    "pos": "noun",
    "def_en": "A person who receives benefits from a trust, will, or insurance policy."
  },
  {
    "en": "bequest",
    "ru": "завещанное наследство",
    "pos": "noun",
    "def_en": "A gift of personal property made in a will."
  },
  {
    "en": "beyond a reasonable doubt",
    "ru": "вне всякого сомнения",
    "pos": "phrase",
    "def_en": "The standard of proof required in criminal cases, requiring the jury to be fully convinced of the defendant's guilt."
  },
  {
    "en": "bifurcation",
    "ru": "раздвоение; раздельное рассмотрение",
    "pos": "noun",
    "def_en": "The division of a trial into separate phases, such as guilt and punishment."
  },
  {
    "en": "bill of indictment",
    "ru": "обвинительное заключение",
    "pos": "phrase",
    "def_en": "A formal written accusation charging a person with a crime, presented to a grand jury."
  },
  {
    "en": "binding agreement",
    "ru": "юридически обязывающее соглашение",
    "pos": "phrase",
    "def_en": "A contract or agreement that is legally enforceable."
  },
  {
    "en": "blackmail",
    "ru": "шантаж; вымогательство",
    "pos": "noun",
    "def_en": "The demand for money or other benefit by threat of revealing damaging information."
  },
  {
    "en": "breach of contract",
    "ru": "нарушение договора",
    "pos": "phrase",
    "def_en": "The failure to perform any term of a contract without a legitimate legal excuse."
  },
  {
    "en": "bribery",
    "ru": "взяточничество; подкуп",
    "pos": "noun",
    "def_en": "The offering, giving, receiving, or soliciting of something of value to influence official action."
  },
  {
    "en": "burden of proof",
    "ru": "бремя доказывания",
    "pos": "phrase",
    "def_en": "The obligation to prove allegations in a legal proceeding."
  },
  {
    "en": "burglary",
    "ru": "кража со взломом",
    "pos": "noun",
    "def_en": "The unlawful entry into a building with the intent to commit a felony or theft."
  },
  {
    "en": "capital crime",
    "ru": "преступление, наказуемое смертной казнью",
    "pos": "phrase",
    "def_en": "A crime for which the punishment may be death."
  },
  {
    "en": "carjacking",
    "ru": "угон автомобиля с применением силы",
    "pos": "noun",
    "def_en": "The forcible theft of a motor vehicle from its driver."
  },
  {
    "en": "case law",
    "ru": "прецедентное право",
    "pos": "phrase",
    "def_en": "Law established by previous court decisions rather than by statutes."
  },
  {
    "en": "caveat emptor",
    "ru": "пусть покупатель будет осторожен",
    "pos": "phrase",
    "def_en": "A principle that the buyer alone is responsible for checking the quality and suitability of goods before a purchase."
  },
  {
    "en": "cease and desist order",
    "ru": "приказ о прекращении действий",
    "pos": "phrase",
    "def_en": "An order issued by a court or agency to stop an illegal or harmful activity."
  },
  {
    "en": "certiorari",
    "ru": "апеллирование в Верховный Суд США",
    "pos": "noun",
    "def_en": "A writ by which a higher court reviews the record of a lower court."
  },
  {
    "en": "chain of custody",
    "ru": "метод прослеживания доказательства",
    "pos": "phrase",
    "def_en": "The documented sequence of persons who had custody of evidence to ensure its integrity."
  },
  {
    "en": "challenge",
    "ru": "отвод (присяжного)",
    "pos": "noun",
    "def_en": "An objection to the seating of a juror, either for cause or peremptorily."
  },
  {
    "en": "charge",
    "ru": "обвинение",
    "pos": "noun",
    "def_en": "A formal accusation of a crime."
  },
  {
    "en": "chattel",
    "ru": "движимое имущество",
    "pos": "noun",
    "def_en": "An article of personal property."
  },
  {
    "en": "check",
    "ru": "сдержка; контроль",
    "pos": "noun",
    "def_en": "A restraint or control, as in checks and balances."
  },
  {
    "en": "circumstantial evidence",
    "ru": "косвенные доказательства",
    "pos": "phrase",
    "def_en": "Evidence that indirectly proves a fact by inference, rather than directly."
  },
  {
    "en": "citation",
    "ru": "распоряжение суда о предъявлении обвинений",
    "pos": "noun",
    "def_en": "A court order to appear, or a reference to a legal authority."
  },
  {
    "en": "civil action",
    "ru": "гражданский иск",
    "pos": "phrase",
    "def_en": "A lawsuit brought to enforce, redress, or protect private rights."
  },
  {
    "en": "claim",
    "ru": "иск; претензия",
    "pos": "noun",
    "def_en": "A demand for a legal remedy or right."
  },
  {
    "en": "class action",
    "ru": "групповой иск",
    "pos": "phrase",
    "def_en": "A lawsuit filed by one or more persons on behalf of a larger group with common interests."
  },
  {
    "en": "clean hands",
    "ru": "чистые руки",
    "pos": "phrase",
    "def_en": "A legal doctrine that a party seeking equitable relief must not have acted unethically."
  },
  {
    "en": "clear and convincing evidence",
    "ru": "явное и убедительное доказательство",
    "pos": "phrase",
    "def_en": "A standard of proof higher than preponderance of the evidence but lower than beyond a reasonable doubt."
  },
  {
    "en": "clerical error",
    "ru": "канцелярская опечатка",
    "pos": "phrase",
    "def_en": "A minor mistake in a legal document, often correctable by the court."
  },
  {
    "en": "codicil",
    "ru": "дополнение к завещанию",
    "pos": "noun",
    "def_en": "A supplement or addition to a will that modifies or revokes provisions."
  },
  {
    "en": "coercion",
    "ru": "принуждение",
    "pos": "noun",
    "def_en": "The use of force or threats to compel someone to act against their will."
  },
  {
    "en": "collateral",
    "ru": "залоговое обеспечение",
    "pos": "noun",
    "def_en": "Property pledged as security for a debt."
  },
  {
    "en": "common law",
    "ru": "общее право",
    "pos": "phrase",
    "def_en": "Law derived from judicial decisions and custom, rather than from statutes."
  },
  {
    "en": "community property",
    "ru": "совместная собственность супругов",
    "pos": "phrase",
    "def_en": "Property acquired during marriage that is owned jointly by both spouses."
  },
  {
    "en": "commutation",
    "ru": "изменение приговора",
    "pos": "noun",
    "def_en": "The reduction of a criminal sentence to a less severe one."
  },
  {
    "en": "complainant",
    "ru": "истец; заявитель",
    "pos": "noun",
    "def_en": "The party who initiates a legal complaint."
  },
  {
    "en": "complaint",
    "ru": "исковое заявление",
    "pos": "noun",
    "def_en": "The initial pleading by a plaintiff setting out the claims against a defendant."
  },
  {
    "en": "concurrent sentences",
    "ru": "совпадающие приговоры",
    "pos": "phrase",
    "def_en": "Sentences for multiple crimes that are served at the same time."
  },
  {
    "en": "concurring opinion",
    "ru": "совпадающее мнение",
    "pos": "phrase",
    "def_en": "A separate opinion by a judge who agrees with the majority's decision but for different reasons."
  },
  {
    "en": "condemnation",
    "ru": "законное отчуждение собственности",
    "pos": "noun",
    "def_en": "The legal process by which the government takes private property for public use with just compensation."
  },
  {
    "en": "confession",
    "ru": "признание",
    "pos": "noun",
    "def_en": "A statement by a person acknowledging their guilt in a crime."
  },
  {
    "en": "conflict of interest",
    "ru": "конфликт интересов",
    "pos": "phrase",
    "def_en": "A situation where a person's personal interests conflict with their professional or public duties."
  },
  {
    "en": "consent",
    "ru": "согласие",
    "pos": "noun",
    "def_en": "Voluntary agreement to an act or proposal."
  },
  {
    "en": "consecutive sentences",
    "ru": "последовательные приговоры",
    "pos": "phrase",
    "def_en": "Sentences for multiple crimes that are served one after the other."
  },
  {
    "en": "conservator",
    "ru": "опекун; попечитель",
    "pos": "noun",
    "def_en": "A person appointed by a court to manage the affairs of an incapacitated individual."
  },
  {
    "en": "constitutional right",
    "ru": "конституционное право",
    "pos": "phrase",
    "def_en": "A right guaranteed by a constitution."
  },
  {
    "en": "constructive possession",
    "ru": "подразумеваемое владение",
    "pos": "phrase",
    "def_en": "Control over property without actual physical possession."
  },
  {
    "en": "contempt of court",
    "ru": "неуважение к суду",
    "pos": "phrase",
    "def_en": "Disobedience or disrespect toward a court, punishable by fine or imprisonment."
  },
  {
    "en": "contingency fee",
    "ru": "гонорар по результату",
    "pos": "phrase",
    "def_en": "A fee paid to a lawyer only if the case is won, usually a percentage of the recovery."
  },
  {
    "en": "contract",
    "ru": "договор",
    "pos": "noun",
    "def_en": "A legally binding agreement between two or more parties."
  },
  {
    "en": "contributory negligence",
    "ru": "встречная небрежность",
    "pos": "phrase",
    "def_en": "A defense that reduces a plaintiff's recovery if they contributed to their own injury."
  },
  {
    "en": "conversion",
    "ru": "незаконное присвоение",
    "pos": "noun",
    "def_en": "The unauthorized taking or use of another's property."
  },
  {
    "en": "conviction",
    "ru": "осуждение; обвинительный приговор",
    "pos": "noun",
    "def_en": "A formal declaration that someone is guilty of a criminal offense."
  },
  {
    "en": "copyright",
    "ru": "авторское право",
    "pos": "noun",
    "def_en": "The exclusive legal right to reproduce, publish, and sell a literary, musical, or artistic work."
  },
  {
    "en": "corroborating evidence",
    "ru": "подтверждающее доказательство",
    "pos": "phrase",
    "def_en": "Evidence that strengthens or confirms other evidence."
  },
  {
    "en": "counsel",
    "ru": "адвокат; юрисконсульт",
    "pos": "noun",
    "def_en": "A lawyer or legal advisor."
  },
  {
    "en": "counterclaim",
    "ru": "встречный иск",
    "pos": "noun",
    "def_en": "A claim brought by a defendant against the plaintiff in the same lawsuit."
  },
  {
    "en": "court of appeals",
    "ru": "апелляционный суд",
    "pos": "phrase",
    "def_en": "An intermediate appellate court that reviews decisions of lower courts."
  },
  {
    "en": "covenant",
    "ru": "договор; обязательство",
    "pos": "noun",
    "def_en": "A formal agreement or promise, often under seal."
  },
  {
    "en": "credibility",
    "ru": "достоверность",
    "pos": "noun",
    "def_en": "The quality of being believable or trustworthy, especially of a witness."
  },
  {
    "en": "crime",
    "ru": "преступление",
    "pos": "noun",
    "def_en": "An act or omission prohibited by law and punishable by the state."
  },
  {
    "en": "criminal intent",
    "ru": "преступный умысел",
    "pos": "phrase",
    "def_en": "The mental state required for a person to be guilty of a crime."
  },
  {
    "en": "cross-examination",
    "ru": "перекрестный допрос",
    "pos": "noun",
    "def_en": "The questioning of a witness by the opposing party to test the truth of their testimony."
  },
  {
    "en": "custody",
    "ru": "опека; содержание под стражей",
    "pos": "noun",
    "def_en": "The care and control of a person or property; also, legal confinement."
  },
  {
    "en": "damages",
    "ru": "возмещение ущерба; убытки",
    "pos": "noun",
    "def_en": "Monetary compensation awarded to a party for loss or injury caused by another."
  },
  {
    "en": "de facto",
    "ru": "де-факто; фактически",
    "pos": "phrase",
    "def_en": "In fact or in practice, whether or not legally recognized."
  },
  {
    "en": "deadly weapon",
    "ru": "смертоносное оружие",
    "pos": "phrase",
    "def_en": "Any object or instrument likely to cause death or serious bodily injury."
  },
  {
    "en": "death penalty",
    "ru": "смертная казнь",
    "pos": "phrase",
    "def_en": "The punishment of execution for a capital crime."
  },
  {
    "en": "debtor",
    "ru": "должник",
    "pos": "noun",
    "def_en": "A person or entity that owes money to another."
  },
  {
    "en": "decree",
    "ru": "судебное решение",
    "pos": "noun",
    "def_en": "A court order or judgment."
  },
  {
    "en": "deed",
    "ru": "договор за печатью",
    "pos": "noun",
    "def_en": "A written instrument that transfers title to real property."
  },
  {
    "en": "defamation",
    "ru": "диффамация; клевета",
    "pos": "noun",
    "def_en": "The publication of a false statement that harms another's reputation."
  },
  {
    "en": "default judgment",
    "ru": "заочное решение",
    "pos": "phrase",
    "def_en": "A judgment entered against a party who fails to appear or respond in a lawsuit."
  },
  {
    "en": "defendant",
    "ru": "ответчик; подсудимый",
    "pos": "noun",
    "def_en": "The party against whom a lawsuit or criminal charge is brought."
  },
  {
    "en": "defense attorney",
    "ru": "адвокат защиты",
    "pos": "phrase",
    "def_en": "A lawyer who represents a defendant in a criminal or civil case."
  },
  {
    "en": "deficiency",
    "ru": "задолженность; недостача",
    "pos": "noun",
    "def_en": "A shortage or lack, especially of money owed."
  },
  {
    "en": "defraud",
    "ru": "обманывать; мошенничать",
    "pos": "verb",
    "def_en": "To deprive someone of property or a right by deception."
  },
  {
    "en": "deposition",
    "ru": "свидетельские показания под присягой",
    "pos": "noun",
    "def_en": "Sworn testimony given out of court and recorded in writing for use in litigation."
  },
  {
    "en": "deterrence",
    "ru": "сдерживание",
    "pos": "noun",
    "def_en": "The use of punishment to discourage future criminal behavior."
  },
  {
    "en": "direct evidence",
    "ru": "прямое доказательство",
    "pos": "phrase",
    "def_en": "Evidence that directly proves a fact without inference."
  },
  {
    "en": "direct examination",
    "ru": "первоначальный допрос",
    "pos": "phrase",
    "def_en": "The first questioning of a witness by the party who called them."
  },
  {
    "en": "discovery",
    "ru": "раскрытие доказательств",
    "pos": "noun",
    "def_en": "The pretrial process by which parties exchange information and evidence."
  },
  {
    "en": "dissent",
    "ru": "особое мнение",
    "pos": "noun",
    "def_en": "A disagreement with the majority opinion in a court decision."
  },
  {
    "en": "divorce",
    "ru": "развод; расторжение брака",
    "pos": "noun",
    "def_en": "The legal dissolution of a marriage."
  },
  {
    "en": "docket",
    "ru": "список судебных дел",
    "pos": "noun",
    "def_en": "A court's calendar of scheduled cases."
  },
  {
    "en": "doctrine",
    "ru": "доктрина; принцип",
    "pos": "noun",
    "def_en": "A legal principle or rule established through judicial decisions or scholarly writing."
  },
  {
    "en": "domicile",
    "ru": "постоянное место жительства",
    "pos": "noun",
    "def_en": "A person's permanent legal residence."
  },
  {
    "en": "double jeopardy",
    "ru": "повторное привлечение к ответственности",
    "pos": "phrase",
    "def_en": "The constitutional protection against being tried twice for the same offense."
  },
  {
    "en": "due process",
    "ru": "надлежащая правовая процедура",
    "pos": "phrase",
    "def_en": "The fair treatment through the judicial system, including notice and an opportunity to be heard."
  },
  {
    "en": "duress",
    "ru": "принуждение",
    "pos": "noun",
    "def_en": "Unlawful pressure that compels a person to act against their will."
  },
  {
    "en": "easement",
    "ru": "сервитут; право пользования чужой собственностью",
    "pos": "noun",
    "def_en": "A legal right to use another's land for a specific purpose."
  },
  {
    "en": "embezzlement",
    "ru": "присвоение или растрата имущества",
    "pos": "noun",
    "def_en": "The fraudulent appropriation of property by a person to whom it was entrusted."
  },
  {
    "en": "eminent domain",
    "ru": "право государства на принудительное отчуждение",
    "pos": "phrase",
    "def_en": "The government's power to take private property for public use with just compensation."
  },
  {
    "en": "en banc",
    "ru": "всем составом суда",
    "pos": "phrase",
    "def_en": "The hearing of a case by all judges of an appellate court, rather than a panel."
  },
  {
    "en": "enjoin",
    "ru": "запретить; вменить в обязанность",
    "pos": "verb",
    "def_en": "To order a person to do or refrain from doing a specific act."
  },
  {
    "en": "entrapment",
    "ru": "провокация преступления",
    "pos": "noun",
    "def_en": "A defense that law enforcement induced a person to commit a crime they would not otherwise have committed."
  },
  {
    "en": "equity",
    "ru": "право справедливости",
    "pos": "noun",
    "def_en": "A system of law based on fairness and justice, supplementing common law."
  },
  {
    "en": "escrow",
    "ru": "условное депонирование",
    "pos": "noun",
    "def_en": "Money or documents held by a neutral third party until a condition is met."
  },
  {
    "en": "estate",
    "ru": "наследственное имущество",
    "pos": "noun",
    "def_en": "All property owned by a person at death, subject to probate."
  },
  {
    "en": "estoppel",
    "ru": "лишение права противоречить",
    "pos": "noun",
    "def_en": "A legal principle preventing a person from contradicting a previous statement or act."
  },
  {
    "en": "evidence",
    "ru": "доказательство",
    "pos": "noun",
    "def_en": "Any matter presented to the court to prove or disprove a fact."
  },
  {
    "en": "ex parte",
    "ru": "в одностороннем порядке",
    "pos": "phrase",
    "def_en": "A proceeding brought by one party without notice to or presence of the other."
  },
  {
    "en": "exclusionary rule",
    "ru": "правило об исключении доказательств",
    "pos": "phrase",
    "def_en": "A rule that evidence obtained in violation of the Constitution is inadmissible."
  },
  {
    "en": "exclusive jurisdiction",
    "ru": "исключительная подсудность",
    "pos": "phrase",
    "def_en": "The power of a court to hear a case to the exclusion of all other courts."
  },
  {
    "en": "exculpatory evidence",
    "ru": "оправдательное доказательство",
    "pos": "phrase",
    "def_en": "Evidence that tends to clear a defendant from guilt."
  },
  {
    "en": "executor",
    "ru": "душеприказчик",
    "pos": "noun",
    "def_en": "A person named in a will to administer the estate of the deceased."
  },
  {
    "en": "exemplary damages",
    "ru": "штрафные убытки",
    "pos": "phrase",
    "def_en": "Damages awarded to punish the defendant and deter similar conduct."
  },
  {
    "en": "expert witness",
    "ru": "свидетель-эксперт",
    "pos": "phrase",
    "def_en": "A witness qualified by knowledge or experience to testify on a specialized subject."
  },
  {
    "en": "extradition",
    "ru": "выдача преступника",
    "pos": "noun",
    "def_en": "The formal process by which one state surrenders a person accused of a crime to another state."
  },
  {
    "en": "felony",
    "ru": "уголовное преступление; фелония",
    "pos": "noun",
    "def_en": "A serious crime, typically punishable by imprisonment for more than one year."
  },
  {
    "en": "fiduciary",
    "ru": "доверенное лицо; фидуциар",
    "pos": "noun",
    "def_en": "A person who acts on behalf of another in a relationship of trust and confidence."
  },
  {
    "en": "filing",
    "ru": "подача документов в суд",
    "pos": "noun",
    "def_en": "The act of submitting a document to a court for inclusion in the record."
  },
  {
    "en": "fine",
    "ru": "штраф",
    "pos": "noun",
    "def_en": "A monetary penalty imposed for a violation of law."
  },
  {
    "en": "firearm",
    "ru": "огнестрельное оружие",
    "pos": "noun",
    "def_en": "A weapon capable of firing a projectile by the action of an explosive."
  },
  {
    "en": "foreclosure",
    "ru": "обращение взыскания на заложенное имущество",
    "pos": "noun",
    "def_en": "The legal process by which a lender takes possession of a property after the borrower defaults."
  },
  {
    "en": "forgery",
    "ru": "подделка документа",
    "pos": "noun",
    "def_en": "The fraudulent making or alteration of a written document."
  },
  {
    "en": "fraud",
    "ru": "мошенничество",
    "pos": "noun",
    "def_en": "Intentional deception to secure unfair or unlawful gain."
  },
  {
    "en": "garnishment",
    "ru": "удержание из заработной платы",
    "pos": "noun",
    "def_en": "A legal process by which a portion of a person's wages is withheld to pay a debt."
  },
  {
    "en": "grand jury",
    "ru": "большое жюри",
    "pos": "phrase",
    "def_en": "A jury that determines whether there is sufficient evidence to charge a person with a crime."
  },
  {
    "en": "grievance",
    "ru": "жалоба",
    "pos": "noun",
    "def_en": "A formal complaint about a violation of rights or entitlements."
  },
  {
    "en": "guilty plea",
    "ru": "признание вины",
    "pos": "phrase",
    "def_en": "A defendant's formal admission in court that they committed the crime charged."
  },
  {
    "en": "habeas corpus",
    "ru": "судебный приказ о доставлении лица в суд",
    "pos": "phrase",
    "def_en": "A writ requiring a person under arrest to be brought before a judge to determine the legality of their detention."
  },
  {
    "en": "hearsay",
    "ru": "показания с чужих слов",
    "pos": "noun",
    "def_en": "An out-of-court statement offered to prove the truth of the matter asserted, generally inadmissible."
  },
  {
    "en": "homicide",
    "ru": "убийство",
    "pos": "noun",
    "def_en": "The killing of one human being by another."
  },
  {
    "en": "hostile witness",
    "ru": "свидетель, дающий показания против вызвавшей стороны",
    "pos": "phrase",
    "def_en": "A witness whose testimony is adverse to the party who called them."
  },
  {
    "en": "immunity",
    "ru": "иммунитет",
    "pos": "noun",
    "def_en": "Exemption from legal liability or prosecution."
  },
  {
    "en": "impeachment",
    "ru": "импичмент",
    "pos": "noun",
    "def_en": "The process of charging a public official with misconduct in office."
  },
  {
    "en": "implied contract",
    "ru": "подразумеваемый договор",
    "pos": "phrase",
    "def_en": "A contract inferred from the conduct of the parties rather than express words."
  },
  {
    "en": "in camera",
    "ru": "в закрытом судебном заседании",
    "pos": "phrase",
    "def_en": "A private hearing held in the judge's chambers, without the public or jury."
  },
  {
    "en": "incarceration",
    "ru": "тюремное заключение",
    "pos": "noun",
    "def_en": "Imprisonment."
  },
  {
    "en": "indictment",
    "ru": "обвинительное заключение",
    "pos": "noun",
    "def_en": "A formal written accusation of a crime, issued by a grand jury."
  },
  {
    "en": "injunction",
    "ru": "судебный запрет",
    "pos": "noun",
    "def_en": "A court order requiring a party to do or refrain from doing a specific act."
  },
  {
    "en": "inquest",
    "ru": "дознание",
    "pos": "noun",
    "def_en": "A judicial inquiry, especially into a death."
  },
  {
    "en": "interrogatories",
    "ru": "письменные вопросы для допроса",
    "pos": "noun",
    "def_en": "Written questions submitted by one party to another as part of discovery."
  },
  {
    "en": "intestate",
    "ru": "умерший без завещания",
    "pos": "noun",
    "def_en": "A person who dies without a valid will."
  },
  {
    "en": "irreparable harm",
    "ru": "непоправимый вред",
    "pos": "phrase",
    "def_en": "Injury that cannot be adequately compensated by money damages."
  },
  {
    "en": "jeopardy",
    "ru": "опасность судебного преследования",
    "pos": "noun",
    "def_en": "The risk of conviction in a criminal proceeding."
  },
  {
    "en": "judgment",
    "ru": "судебное решение",
    "pos": "noun",
    "def_en": "A court's final determination of the rights and claims of the parties."
  },
  {
    "en": "judicial review",
    "ru": "судебный надзор",
    "pos": "phrase",
    "def_en": "The power of courts to review and invalidate laws or acts of government that violate the constitution."
  },
  {
    "en": "jurisdiction",
    "ru": "юрисдикция; подсудность",
    "pos": "noun",
    "def_en": "The authority of a court to hear and decide a case."
  },
  {
    "en": "jurisprudence",
    "ru": "юриспруденция",
    "pos": "noun",
    "def_en": "The theory or philosophy of law."
  },
  {
    "en": "juvenile court",
    "ru": "суд по делам несовершеннолетних",
    "pos": "phrase",
    "def_en": "A court that deals with offenses committed by minors."
  },
  {
    "en": "kidnapping",
    "ru": "похищение человека",
    "pos": "noun",
    "def_en": "The unlawful taking and carrying away of a person by force or fraud."
  },
  {
    "en": "larceny",
    "ru": "кража; хищение чужого имущества",
    "pos": "noun",
    "def_en": "The unlawful taking and carrying away of personal property with intent to permanently deprive the owner."
  },
  {
    "en": "lawsuit",
    "ru": "гражданский иск",
    "pos": "noun",
    "def_en": "A legal proceeding in a civil court."
  },
  {
    "en": "leading question",
    "ru": "наводящий вопрос",
    "pos": "phrase",
    "def_en": "A question that suggests the answer desired by the examiner."
  },
  {
    "en": "lease",
    "ru": "договор аренды",
    "pos": "noun",
    "def_en": "A contract granting possession and use of property for a specified period in exchange for rent."
  },
  {
    "en": "legal aid",
    "ru": "бесплатная юридическая помощь",
    "pos": "phrase",
    "def_en": "Government-funded legal assistance for those who cannot afford a lawyer."
  },
  {
    "en": "legal capacity",
    "ru": "правоспособность",
    "pos": "phrase",
    "def_en": "A person's legal ability to enter into contracts and sue or be sued."
  },
  {
    "en": "legislation",
    "ru": "законодательство",
    "pos": "noun",
    "def_en": "The process of making laws; also, the laws themselves."
  },
  {
    "en": "legislature",
    "ru": "законодательный орган",
    "pos": "noun",
    "def_en": "A body of persons empowered to make laws."
  },
  {
    "en": "libel",
    "ru": "клевета в письменной форме",
    "pos": "noun",
    "def_en": "A defamatory statement published in writing or other permanent form."
  },
  {
    "en": "lien",
    "ru": "залоговое право",
    "pos": "noun",
    "def_en": "A legal right to hold another's property as security for a debt."
  },
  {
    "en": "limitation of actions",
    "ru": "срок исковой давности",
    "pos": "phrase",
    "def_en": "A statute setting the maximum time after an event within which legal proceedings may be initiated."
  },
  {
    "en": "litigation",
    "ru": "судебный процесс; тяжба",
    "pos": "noun",
    "def_en": "The process of taking legal action; a lawsuit."
  },
  {
    "en": "magistrate",
    "ru": "магистрат; мировой судья",
    "pos": "noun",
    "def_en": "A civil officer who administers the law, especially one who conducts preliminary hearings."
  },
  {
    "en": "maintenance",
    "ru": "алименты; содержание",
    "pos": "noun",
    "def_en": "Financial support paid by one spouse to another, or for a child."
  },
  {
    "en": "malice aforethought",
    "ru": "заранее обдуманный злой умысел",
    "pos": "phrase",
    "def_en": "The mental state required for murder, showing intent to kill or cause grievous harm."
  },
  {
    "en": "manslaughter",
    "ru": "непредумышленное убийство",
    "pos": "noun",
    "def_en": "The unlawful killing of a human being without malice aforethought."
  },
  {
    "en": "material evidence",
    "ru": "существенное доказательство",
    "pos": "phrase",
    "def_en": "Evidence that is relevant and has a tendency to influence the outcome of a case."
  },
  {
    "en": "mediation",
    "ru": "медиация",
    "pos": "noun",
    "def_en": "A process in which a neutral third party helps disputing parties reach a settlement."
  },
  {
    "en": "mens rea",
    "ru": "виновная воля; вина",
    "pos": "noun",
    "def_en": "The mental element of a crime; the guilty mind."
  },
  {
    "en": "misdemeanor",
    "ru": "проступок; мисдиминор",
    "pos": "noun",
    "def_en": "A minor crime punishable by fine or imprisonment for less than one year."
  },
  {
    "en": "mistrial",
    "ru": "аннуляция судебного процесса",
    "pos": "noun",
    "def_en": "A trial that is terminated before a verdict due to a fundamental error or deadlocked jury."
  },
  {
    "en": "mitigating circumstances",
    "ru": "смягчающие обстоятельства",
    "pos": "phrase",
    "def_en": "Facts that may reduce the severity of a punishment but do not excuse the crime."
  },
  {
    "en": "moot",
    "ru": "спорный; не имеющий практического значения",
    "pos": "adjective",
    "def_en": "Subject to debate; also, a point that is no longer relevant to a case."
  },
  {
    "en": "moral turpitude",
    "ru": "безнравственность",
    "pos": "phrase",
    "def_en": "Conduct that is contrary to community standards of justice, honesty, or good morals."
  },
  {
    "en": "mortgage",
    "ru": "ипотека",
    "pos": "noun",
    "def_en": "A loan secured by real property."
  },
  {
    "en": "motion",
    "ru": "ходатайство",
    "pos": "noun",
    "def_en": "A formal request to a court for a ruling or order."
  },
  {
    "en": "murder",
    "ru": "умышленное убийство",
    "pos": "noun",
    "def_en": "The unlawful killing of a human being with malice aforethought."
  },
  {
    "en": "negligence",
    "ru": "небрежность",
    "pos": "noun",
    "def_en": "Failure to exercise the care that a reasonably prudent person would exercise."
  },
  {
    "en": "negotiable instrument",
    "ru": "оборотный документ",
    "pos": "phrase",
    "def_en": "A written document signed by the maker, promising to pay a sum of money."
  },
  {
    "en": "no-fault",
    "ru": "без вины",
    "pos": "phrase",
    "def_en": "A legal system in which fault is not considered, as in no-fault divorce or insurance."
  },
  {
    "en": "notary public",
    "ru": "нотариус",
    "pos": "phrase",
    "def_en": "A public officer authorized to administer oaths and certify documents."
  },
  {
    "en": "nuisance",
    "ru": "источник вреда; нарушение покоя",
    "pos": "noun",
    "def_en": "An unreasonable interference with the use and enjoyment of property."
  },
  {
    "en": "oath",
    "ru": "присяга",
    "pos": "noun",
    "def_en": "A solemn promise to tell the truth, often invoking a divine witness."
  },
  {
    "en": "objection",
    "ru": "возражение",
    "pos": "noun",
    "def_en": "A formal protest to the admission of evidence or a question."
  },
  {
    "en": "offense",
    "ru": "правонарушение; преступление",
    "pos": "noun",
    "def_en": "A violation of law."
  },
  {
    "en": "offer",
    "ru": "оферта; предложение",
    "pos": "noun",
    "def_en": "A proposal to enter into a contract."
  },
  {
    "en": "opening statement",
    "ru": "вступительная речь",
    "pos": "phrase",
    "def_en": "A statement by counsel at the beginning of a trial outlining the facts they intend to prove."
  },
  {
    "en": "opinion",
    "ru": "мнение суда",
    "pos": "noun",
    "def_en": "The written explanation of a court's decision."
  },
  {
    "en": "ordinance",
    "ru": "постановление местного органа власти",
    "pos": "noun",
    "def_en": "A law enacted by a local government."
  },
  {
    "en": "overt act",
    "ru": "явное действие",
    "pos": "phrase",
    "def_en": "An open act showing an intent to commit a crime."
  },
  {
    "en": "pardon",
    "ru": "помилование",
    "pos": "noun",
    "def_en": "An act of grace by a governor or president releasing a convicted person from punishment."
  },
  {
    "en": "parole",
    "ru": "условно-досрочное освобождение",
    "pos": "noun",
    "def_en": "The release of a prisoner before the expiration of their sentence under supervision."
  },
  {
    "en": "party",
    "ru": "сторона (в судебном процессе)",
    "pos": "noun",
    "def_en": "A person or entity involved in a legal proceeding."
  },
  {
    "en": "penalty",
    "ru": "наказание; санкция",
    "pos": "noun",
    "def_en": "A punishment imposed for a violation of law."
  },
  {
    "en": "per curiam",
    "ru": "решением суда",
    "pos": "phrase",
    "def_en": "An opinion issued by an appellate court as a whole, without attributing authorship."
  },
  {
    "en": "peremptory challenge",
    "ru": "немотивированный отвод",
    "pos": "phrase",
    "def_en": "The right to reject a potential juror without stating a reason."
  },
  {
    "en": "perjury",
    "ru": "лжесвидетельство",
    "pos": "noun",
    "def_en": "The crime of intentionally lying under oath."
  },
  {
    "en": "petition",
    "ru": "исковое заявление; петиция",
    "pos": "noun",
    "def_en": "A formal written request to a court."
  },
  {
    "en": "plaintiff",
    "ru": "истец",
    "pos": "noun",
    "def_en": "The party who initiates a lawsuit."
  },
  {
    "en": "plea bargain",
    "ru": "сделка о признании вины",
    "pos": "phrase",
    "def_en": "An agreement between the prosecutor and defendant in which the defendant pleads guilty to a lesser charge."
  },
  {
    "en": "plea",
    "ru": "заявление (в суде)",
    "pos": "noun",
    "def_en": "A defendant's formal answer to a criminal charge."
  },
  {
    "en": "pleadings",
    "ru": "состязательные бумаги",
    "pos": "noun",
    "def_en": "Formal written statements of the parties' claims and defenses."
  },
  {
    "en": "police power",
    "ru": "полицейская власть",
    "pos": "phrase",
    "def_en": "The inherent authority of a government to regulate for the health, safety, and welfare of the public."
  },
  {
    "en": "precedent",
    "ru": "прецедент",
    "pos": "noun",
    "def_en": "A prior court decision that serves as a rule for future similar cases."
  },
  {
    "en": "preliminary hearing",
    "ru": "предварительное слушание",
    "pos": "phrase",
    "def_en": "A hearing to determine if there is sufficient evidence to hold a defendant for trial."
  },
  {
    "en": "preliminary injunction",
    "ru": "предварительный судебный запрет",
    "pos": "phrase",
    "def_en": "A court order issued before a trial to prevent harm until the case is decided."
  },
  {
    "en": "premeditation",
    "ru": "предварительное обдумывание",
    "pos": "noun",
    "def_en": "The act of planning or deliberating a crime before committing it."
  },
  {
    "en": "preponderance of the evidence",
    "ru": "перевес доказательств",
    "pos": "phrase",
    "def_en": "The standard of proof in civil cases, meaning the evidence is more likely true than not."
  },
  {
    "en": "presumption of innocence",
    "ru": "презумпция невиновности",
    "pos": "phrase",
    "def_en": "The principle that a defendant is considered innocent until proven guilty."
  },
  {
    "en": "pre-trial conference",
    "ru": "предварительное судебное совещание",
    "pos": "phrase",
    "def_en": "A meeting between parties and the judge before trial to narrow issues and discuss settlement."
  },
  {
    "en": "prima facie",
    "ru": "на первый взгляд; при отсутствии опровержения",
    "pos": "phrase",
    "def_en": "Evidence that is sufficient on its face to prove a fact unless rebutted."
  },
  {
    "en": "probable cause",
    "ru": "достаточное основание",
    "pos": "phrase",
    "def_en": "Reasonable grounds to believe a crime has been committed, justifying arrest or search."
  },
  {
    "en": "probate",
    "ru": "утверждение завещания",
    "pos": "noun",
    "def_en": "The judicial process of validating a will and administering an estate."
  },
  {
    "en": "probation",
    "ru": "условное осуждение",
    "pos": "noun",
    "def_en": "A sentence allowing a convicted person to remain in the community under supervision."
  },
  {
    "en": "procedural law",
    "ru": "процессуальное право",
    "pos": "phrase",
    "def_en": "The rules governing the conduct of legal proceedings."
  },
  {
    "en": "prosecutor",
    "ru": "прокурор; обвинитель",
    "pos": "noun",
    "def_en": "The public official who conducts criminal prosecutions."
  },
  {
    "en": "proximate cause",
    "ru": "непосредственная причина",
    "pos": "phrase",
    "def_en": "The primary cause of an injury; a cause that is legally sufficient to result in liability."
  },
  {
    "en": "punitive damages",
    "ru": "штрафные убытки",
    "pos": "phrase",
    "def_en": "Damages awarded to punish the defendant for egregious conduct."
  },
  {
    "en": "quash",
    "ru": "отменять; аннулировать",
    "pos": "verb",
    "def_en": "To nullify or set aside a legal proceeding or indictment."
  },
  {
    "en": "quid pro quo",
    "ru": "услуга за услугу",
    "pos": "phrase",
    "def_en": "Something given or received in exchange for something else."
  },
  {
    "en": "quo warranto",
    "ru": "судебный приказ о производстве",
    "pos": "phrase",
    "def_en": "A writ requiring a person to show by what authority they hold an office or franchise."
  },
  {
    "en": "rape",
    "ru": "изнасилование",
    "pos": "noun",
    "def_en": "Unlawful sexual intercourse by force or threat of force."
  },
  {
    "en": "reasonable doubt",
    "ru": "разумное сомнение",
    "pos": "phrase",
    "def_en": "The doubt that prevents one from being firmly convinced of a defendant's guilt."
  },
  {
    "en": "rebuttal",
    "ru": "опровержение",
    "pos": "noun",
    "def_en": "Evidence presented to contradict or overcome other evidence."
  },
  {
    "en": "recidivism",
    "ru": "рецидив",
    "pos": "noun",
    "def_en": "The tendency of a convicted criminal to reoffend."
  },
  {
    "en": "reckless driving",
    "ru": "небрежное вождение",
    "pos": "phrase",
    "def_en": "Driving with a willful or wanton disregard for the safety of persons or property."
  },
  {
    "en": "recognizance",
    "ru": "обязательство, данное в суде",
    "pos": "noun",
    "def_en": "A bond by which a person undertakes before a court to do or refrain from doing something."
  },
  {
    "en": "redirect examination",
    "ru": "повторный допрос",
    "pos": "phrase",
    "def_en": "The questioning of a witness by the party who called them, after cross-examination."
  },
  {
    "en": "recusal",
    "ru": "отвод (судьи)",
    "pos": "noun",
    "def_en": "The act of a judge withdrawing from a case due to a conflict of interest."
  },
  {
    "en": "remedy",
    "ru": "средство судебной защиты",
    "pos": "noun",
    "def_en": "The means by which a legal right is enforced or a wrong is redressed."
  },
  {
    "en": "restitution",
    "ru": "восстановление первоначального правового положения",
    "pos": "noun",
    "def_en": "The restoration of property to its rightful owner; also, compensation for loss."
  },
  {
    "en": "restraining order",
    "ru": "запретительный судебный приказ",
    "pos": "phrase",
    "def_en": "A court order that prohibits a person from doing a specific act, often for protection."
  },
  {
    "en": "retainer",
    "ru": "соглашение с адвокатом",
    "pos": "noun",
    "def_en": "The act of engaging a lawyer; also, the fee paid to secure legal services."
  },
  {
    "en": "reverse",
    "ru": "отменять (судебное решение)",
    "pos": "verb",
    "def_en": "To set aside or annul a lower court's judgment."
  },
  {
    "en": "revocation",
    "ru": "отмена; аннулирование",
    "pos": "noun",
    "def_en": "The act of cancelling or annulling something, such as a will or license."
  },
  {
    "en": "robbery",
    "ru": "грабеж; разбой",
    "pos": "noun",
    "def_en": "The taking of property from a person by force or threat of force."
  },
  {
    "en": "rule of law",
    "ru": "верховенство права",
    "pos": "phrase",
    "def_en": "The principle that all persons and institutions are subject to and accountable under the law."
  },
  {
    "en": "sanction",
    "ru": "санкция",
    "pos": "noun",
    "def_en": "A penalty or coercive measure imposed for a violation of law."
  },
  {
    "en": "search warrant",
    "ru": "ордер на обыск",
    "pos": "phrase",
    "def_en": "A court order authorizing law enforcement to search a specific place for evidence."
  },
  {
    "en": "seizure",
    "ru": "выемка; изъятие",
    "pos": "noun",
    "def_en": "The taking of property by legal authority."
  },
  {
    "en": "self-incrimination",
    "ru": "самоизобличение",
    "pos": "noun",
    "def_en": "The act of exposing oneself to criminal prosecution by giving testimony."
  },
  {
    "en": "sentence",
    "ru": "приговор",
    "pos": "noun",
    "def_en": "The punishment imposed by a court on a convicted defendant."
  },
  {
    "en": "sequestration",
    "ru": "изоляция присяжных",
    "pos": "noun",
    "def_en": "The isolation of a jury to prevent outside influence."
  },
  {
    "en": "settlement",
    "ru": "мировое соглашение",
    "pos": "noun",
    "def_en": "An agreement resolving a dispute without a trial."
  },
  {
    "en": "slander",
    "ru": "клевета в устной форме",
    "pos": "noun",
    "def_en": "A defamatory statement spoken aloud."
  },
  {
    "en": "small claims court",
    "ru": "суд по делам с небольшой суммой иска",
    "pos": "phrase",
    "def_en": "A court that handles civil cases involving small amounts of money."
  },
  {
    "en": "solicitation",
    "ru": "подстрекательство",
    "pos": "noun",
    "def_en": "The act of encouraging or requesting another to commit a crime."
  },
  {
    "en": "specific performance",
    "ru": "реальное исполнение",
    "pos": "phrase",
    "def_en": "A remedy requiring a party to perform a contract as agreed."
  },
  {
    "en": "standing",
    "ru": "право на иск",
    "pos": "noun",
    "def_en": "The legal right to bring a lawsuit."
  },
  {
    "en": "stare decisis",
    "ru": "доктрина судебного прецедента",
    "pos": "phrase",
    "def_en": "The principle that courts should follow precedent."
  },
  {
    "en": "statute",
    "ru": "статут; закон",
    "pos": "noun",
    "def_en": "A law enacted by a legislative body."
  },
  {
    "en": "statute of limitations",
    "ru": "срок исковой давности",
    "pos": "phrase",
    "def_en": "A law setting the time limit for filing a lawsuit."
  },
  {
    "en": "stay",
    "ru": "приостановление производства",
    "pos": "noun",
    "def_en": "A court order halting a proceeding."
  },
  {
    "en": "stipulation",
    "ru": "соглашение сторон",
    "pos": "noun",
    "def_en": "An agreement between opposing parties about a fact or issue in a case."
  },
  {
    "en": "subpoena",
    "ru": "повестка в суд",
    "pos": "noun",
    "def_en": "A court order requiring a person to appear as a witness or produce documents."
  },
  {
    "en": "summary judgment",
    "ru": "судебное решение без судебного разбирательства",
    "pos": "phrase",
    "def_en": "A judgment entered without a full trial when there is no genuine dispute of material fact."
  },
  {
    "en": "summons",
    "ru": "судебная повестка",
    "pos": "noun",
    "def_en": "A document informing a defendant of a lawsuit and requiring a response."
  },
  {
    "en": "suppress",
    "ru": "пресекать; запретить",
    "pos": "verb",
    "def_en": "To prevent evidence from being used at trial."
  },
  {
    "en": "surety",
    "ru": "поручитель",
    "pos": "noun",
    "def_en": "A person who takes responsibility for another's debt or obligation."
  },
  {
    "en": "suspended sentence",
    "ru": "условный приговор",
    "pos": "phrase",
    "def_en": "A sentence that is not served if the defendant meets certain conditions."
  },
  {
    "en": "testimony",
    "ru": "свидетельские показания",
    "pos": "noun",
    "def_en": "Evidence given by a witness under oath."
  },
  {
    "en": "tort",
    "ru": "деликт; гражданское правонарушение",
    "pos": "noun",
    "def_en": "A civil wrong that causes a claimant to suffer loss or harm."
  },
  {
    "en": "tortfeasor",
    "ru": "правонарушитель; причинитель вреда",
    "pos": "noun",
    "def_en": "A person who commits a tort."
  },
  {
    "en": "transcript",
    "ru": "протокол судебного заседания",
    "pos": "noun",
    "def_en": "The official written record of a court proceeding."
  },
  {
    "en": "trespass",
    "ru": "нарушение чужого права владения",
    "pos": "noun",
    "def_en": "The unlawful entry onto another's land."
  },
  {
    "en": "trial",
    "ru": "судебное разбирательство",
    "pos": "noun",
    "def_en": "A formal examination of evidence before a court to determine guilt or liability."
  },
  {
    "en": "trust",
    "ru": "доверительная собственность; траст",
    "pos": "noun",
    "def_en": "A legal arrangement in which one person holds property for the benefit of another."
  },
  {
    "en": "trustee",
    "ru": "доверительный собственник",
    "pos": "noun",
    "def_en": "A person who holds property in trust for another."
  },
  {
    "en": "unconstitutional",
    "ru": "неконституционный",
    "pos": "adjective",
    "def_en": "Not in accordance with the constitution."
  },
  {
    "en": "unlawful",
    "ru": "противозаконный",
    "pos": "adjective",
    "def_en": "Not authorized by law; illegal."
  },
  {
    "en": "usury",
    "ru": "взимание процентов выше разрешенного законом",
    "pos": "noun",
    "def_en": "The practice of charging interest rates that are illegally high."
  },
  {
    "en": "venue",
    "ru": "место рассмотрения дела",
    "pos": "noun",
    "def_en": "The geographic location where a case is tried."
  },
  {
    "en": "verdict",
    "ru": "вердикт",
    "pos": "noun",
    "def_en": "The decision of a jury on the factual issues of a case."
  },
  {
    "en": "void",
    "ru": "ничтожный; не имеющий юридической силы",
    "pos": "adjective",
    "def_en": "Having no legal force or effect."
  },
  {
    "en": "voir dire",
    "ru": "предварительная проверка присяжных",
    "pos": "phrase",
    "def_en": "The questioning of potential jurors to determine their suitability."
  },
  {
    "en": "waiver",
    "ru": "отказ (от права)",
    "pos": "noun",
    "def_en": "The intentional relinquishment of a known right."
  },
  {
    "en": "warrant",
    "ru": "ордер",
    "pos": "noun",
    "def_en": "A court order authorizing law enforcement to make an arrest or conduct a search."
  },
  {
    "en": "will",
    "ru": "завещание",
    "pos": "noun",
    "def_en": "A legal document declaring how a person's property is to be distributed after death."
  },
  {
    "en": "with prejudice",
    "ru": "с ущербом для права",
    "pos": "phrase",
    "def_en": "A dismissal of a case that prevents the plaintiff from filing the same claim again."
  },
  {
    "en": "witness",
    "ru": "свидетель",
    "pos": "noun",
    "def_en": "A person who gives testimony in a legal proceeding."
  },
  {
    "en": "writ",
    "ru": "судебный приказ",
    "pos": "noun",
    "def_en": "A formal written order issued by a court."
  },
  {
    "en": "wrongful termination",
    "ru": "неправомерное прекращение обязательств",
    "pos": "phrase",
    "def_en": "The illegal dismissal of an employee in violation of contract or law."
  },
  {
    "en": "yield",
    "ru": "уступать; соглашаться",
    "pos": "verb",
    "def_en": "To give way to a superior claim or right."
  },
  {
    "en": "zero tolerance",
    "ru": "абсолютная недопустимость",
    "pos": "phrase",
    "def_en": "A policy of punishing all violations, no matter how minor."
  }
]

PARALLEL_V5: List[Tuple[str, str]] = [
    ("The defendant's alibi was corroborated by two independent witnesses.",
     "Алиби подсудимого было подтверждено двумя независимыми свидетелями."),
    ("The grand jury returned an indictment charging the defendant with fraud.",
     "Большое жюри вынесло обвинительное заключение, обвинив подсудимого в мошенничестве."),
    ("The suspect was released on bail pending the preliminary hearing.",
     "Подозреваемый был освобождён под залог до предварительного слушания."),
    ("The court issued an arrest warrant after reviewing the affidavit.",
     "Суд выдал ордер на арест после рассмотрения аффидевита."),
    ("The defendant was arraigned on charges of aggravated assault and battery.",
     "Подсудимому было предъявлено обвинение в нападении при отягчающих обстоятельствах и избиении."),
    ("The jury returned a verdict of not guilty after three hours of deliberation.",
     "Жюри вынесло вердикт о невиновности после трёх часов обсуждения."),
    ("The prosecution must prove the defendant's guilt beyond a reasonable doubt.",
     "Обвинение должно доказать вину подсудимого вне всякого сомнения."),
    ("The appellate court overturned the conviction due to a prejudicial error.",
     "Апелляционный суд отменил обвинительный приговор ввиду ошибки, ущемляющей права стороны."),
    ("The accused was acquitted on all counts for lack of admissible evidence.",
     "Обвиняемый был оправдан по всем пунктам обвинения за отсутствием допустимых доказательств."),
    ("The witness testified under oath during the cross-examination.",
     "Свидетель дал показания под присягой во время перекрёстного допроса."),
    ("The defendant entered a plea of guilty to the lesser charge.",
     "Подсудимый признал вину по менее тяжкому обвинению."),
    ("The suspect's testimony was challenged as hearsay by the defense counsel.",
     "Показания подозреваемого были оспорены защитой как показания с чужих слов."),
    ("The judge issued a bench warrant when the defendant failed to appear.",
     "Судья выдал ордер на арест, когда подсудимый не явился в суд."),
    ("The police conducted a search and seizure without a valid warrant.",
     "Полиция провела обыск и изъятие без действительного ордера."),
    ("The defendant invoked his right against self-incrimination.",
     "Подсудимый воспользовался своим правом не свидетельствовать против себя."),
    ("The grand jury refused to return a true bill against the accused.",
     "Большое жюри отказалось утвердить обвинительный акт против обвиняемого."),
    ("The defendant was charged with voluntary manslaughter, not murder.",
     "Подсудимому было предъявлено обвинение в непредумышленном убийстве, а не в умышленном."),
    ("The court found the confession involuntary and excluded it from evidence.",
     "Суд признал признание данным недобровольно и исключил его из доказательств."),
    ("The plaintiff filed a motion for summary judgment in the civil case.",
     "Истец подал ходатайство о вынесении решения без судебного разбирательства по гражданскому делу."),
    ("The defendant filed a counterclaim alleging breach of contract.",
     "Ответчик подал встречный иск, ссылаясь на нарушение договора."),
    ("The plaintiff sought punitive damages for the willful breach of contract.",
     "Истец требовал штрафные убытки за умышленное нарушение договора."),
    ("The parties reached a settlement before the trial began.",
     "Стороны достигли мирового соглашения до начала судебного разбирательства."),
    ("The court granted an injunction preventing the merger of the two companies.",
     "Суд вынес судебный запрет, препятствующий слиянию двух компаний."),
    ("The contract was declared void due to duress and undue influence.",
     "Договор был признан ничтожным вследствие принуждения и неправомерного влияния."),
    ("The plaintiff lacked standing to bring a class action against the manufacturer.",
     "Истец не имел права на подачу группового иска против производителя."),
    ("The court ordered specific performance of the real estate contract.",
     "Суд постановил реальное исполнение договора купли-продажи недвижимости."),
    ("The defendant was held liable for negligence in the slip-and-fall case.",
     "Ответчик был признан виновным в небрежности по делу о падении на скользком полу."),
    ("The arbitrator issued a binding award resolving the commercial dispute.",
     "Арбитр вынес обязательное решение, разрешив коммерческий спор."),
    ("The plaintiff's claim for restitution was denied by the court.",
     "Требование истца о реституции было отклонено судом."),
    ("The parties agreed to resolve their dispute through mediation.",
     "Стороны согласились разрешить свой спор посредством медиации."),
    ("The court awarded compensatory damages for the personal injury.",
     "Суд присудил возмещение ущерба за причинение вреда здоровью."),
    ("The parties entered into a legally binding agreement subject to arbitration.",
     "Стороны заключили юридически обязывающее соглашение с условием об арбитраже."),
    ("The company filed for bankruptcy protection under Chapter 11.",
     "Компания подала заявление о защите от банкротства по Главе 11."),
    ("The escrow agent held the funds until the closing conditions were met.",
     "Эскроу-агент удерживал средства до выполнения условий закрытия сделки."),
    ("The parties signed a non-disclosure agreement before the negotiations.",
     "Стороны подписали соглашение о неразглашении до начала переговоров."),
    ("The corporation's articles of association were amended by shareholder vote.",
     "Устав корпорации был изменён по решению акционеров."),
    ("The borrower defaulted on the mortgage, and foreclosure proceedings began.",
     "Заёмщик допустил просрочку по ипотеке, и началась процедура обращения взыскания."),
    ("The parties agreed to a non-competition clause in the employment contract.",
     "Стороны согласовали условие о неконкуренции в трудовом договоре."),
    ("The testator bequeathed the estate to his two children.",
     "Завещатель завещал имущество двум своим детям."),
    ("The executor of the will filed a petition for probate.",
     "Душеприказчик подал заявление об утверждении завещания."),
    ("The decedent died intestate, and the estate was distributed by law.",
     "Умерший скончался без завещания, и имущество было распределено по закону."),
    ("The trustee managed the trust assets for the benefit of the beneficiaries.",
     "Доверительный собственник управлял активами траста в интересах бенефициаров."),
    ("The court appointed a guardian ad litem for the minor heir.",
     "Суд назначил опекуна для защиты интересов несовершеннолетнего наследника."),
    ("The codicil modified the terms of the original will.",
     "Дополнение к завещанию изменило условия первоначального завещания."),
    ("The court held that the statute violated the equal protection clause.",
     "Суд постановил, что статут нарушает положение о равной защите."),
    ("The defendant claimed a violation of his right to due process.",
     "Подсудимый заявил о нарушении его права на надлежащую правовую процедуру."),
    ("The police action was challenged as an unlawful seizure under the Fourth Amendment.",
     "Действия полиции были оспорены как незаконное изъятие в нарушение Четвёртой поправки."),
    ("The court applied the exclusionary rule to suppress the evidence.",
     "Суд применил правило об исключении доказательств, чтобы не допустить их использования."),
    ("The state requested the extradition of the fugitive from abroad.",
     "Государство запросило экстрадицию беглеца из-за рубежа."),
    ("The parties chose the law of the State of New York as the governing law.",
     "Стороны выбрали право штата Нью-Йорк в качестве применимого права."),
    ("The arbitral tribunal applied the rules of the International Chamber of Commerce.",
     "Арбитражный трибунал применил правила Международной торговой палаты."),
    ("The foreign judgment was recognized and enforced by the domestic court.",
     "Иностранное судебное решение было признано и приведено в исполнение местным судом."),
    ("The plaintiff served the summons and complaint on the defendant.",
     "Истец вручил ответчику судебную повестку и исковое заявление."),
    ("The parties conducted discovery, including depositions and interrogatories.",
     "Стороны провели раскрытие доказательств, включая письменные показания и опросные листы."),
    ("The court granted a motion in limine to exclude the prejudicial evidence.",
     "Суд удовлетворил ходатайство о недопущении предвзятого доказательства."),
    ("The judge denied the motion to suppress the evidence obtained illegally.",
     "Судья отклонил ходатайство об исключении доказательств, полученных незаконно."),
    ("The witness was sequestered to prevent influence from other testimony.",
     "Свидетель был изолирован во избежание влияния других показаний."),
    ("The losing party filed a notice of appeal within the statutory period.",
     "Проигравшая сторона подала уведомление об апелляции в установленный законом срок."),
    ("The court reporter prepared a transcript of the entire proceeding.",
     "Судебный стенографист подготовил протокол всего судебного заседания."),
    ("The appellate court remanded the case for a new trial.",
     "Апелляционный суд направил дело на новое рассмотрение."),
    ("The jury was polled to confirm the verdict was unanimous.",
     "Было проведено индивидуальное голосование присяжных для подтверждения единогласия вердикта."),
    ("The court issued a writ of habeas corpus to review the detention.",
     "Суд выдал судебный приказ о доставлении лица в суд для проверки законности задержания."),
]


def build_tbx(terms: Sequence[Dict[str, str]], path: str,
              subject_field: str = SUBJECT_FIELD) -> str:
    martif = ET.Element("martif", {"type": "TBX", XML_LANG: "en"})

    header = ET.SubElement(martif, "martifHeader")
    file_desc = ET.SubElement(header, "fileDesc")
    title_stmt = ET.SubElement(file_desc, "titleStmt")
    ET.SubElement(title_stmt, "title").text = f"Glossary EN-RU: {subject_field}"
    source_desc = ET.SubElement(file_desc, "sourceDesc")
    ET.SubElement(source_desc, "p").text = "Lab 01, variant 5"

    body = ET.SubElement(ET.SubElement(martif, "text"), "body")

    for index, item in enumerate(terms, start=1):
        entry = ET.SubElement(body, "termEntry", {"id": f"tid-{index:03d}"})
        ET.SubElement(entry, "descrip", {"type": "subjectField"}).text = subject_field

        lang_en = ET.SubElement(entry, "langSet", {XML_LANG: "en"})
        tig = ET.SubElement(lang_en, "tig")
        ET.SubElement(tig, "term").text = item["en"]
        ET.SubElement(tig, "termNote", {"type": "partOfSpeech"}).text = item["pos"]
        ET.SubElement(tig, "descrip", {"type": "definition"}).text = item.get("def_en", "")

        lang_ru = ET.SubElement(entry, "langSet", {XML_LANG: "ru"})
        term_grp = ET.SubElement(ET.SubElement(lang_ru, "ntig"), "termGrp")
        ET.SubElement(term_grp, "term").text = item["ru"]
        ET.SubElement(term_grp, "termNote", {"type": "partOfSpeech"}).text = item["pos"]

    xml_body = ET.tostring(martif, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write(xml_body)
    return path


def build_tmx(pairs: Sequence[Tuple[str, str]], path: str,
              src_lang: str = "en", tgt_lang: str = "ru") -> str:
    tmx = ET.Element("tmx", {"version": "1.4"})
    ET.SubElement(tmx, "header", {
        "creationtool": "lab01-generator",
        "creationtoolversion": "1.0",
        "segtype": "sentence",
        "o-tmf": "TMX",
        "adminlang": "en",
        "srclang": src_lang,
        "datatype": "plaintext",
    })
    body = ET.SubElement(tmx, "body")

    for index, (source, target) in enumerate(pairs, start=1):
        tu = ET.SubElement(body, "tu", {"tuid": str(index)})
        ET.SubElement(tu, "prop", {"type": "domain"}).text = "Law / Jurisprudence"
        for lang, text in ((src_lang, source), (tgt_lang, target)):
            tuv = ET.SubElement(tu, "tuv", {XML_LANG: lang})
            ET.SubElement(tuv, "seg").text = text

    xml_body = ET.tostring(tmx, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write(xml_body)
    return path


build_tbx(TERMS_V5, TBX_FILE)
build_tmx(PARALLEL_V5, TMX_FILE)

my_glossary = parse_tbx(TBX_FILE)
my_tm = parse_tmx(TMX_FILE)

print(f"Глоссарий: {len(my_glossary)} терминов")
print(f"Память переводов: {len(my_tm)} сегментов")

QUERIES_V5 = [
    {
        "label": "1. ICE — точное совпадение с совпадением контекста",
        "query": "The grand jury returned an indictment charging the defendant with fraud.",
        "context": "The suspect was released on bail pending the preliminary hearing.",
    },
    {
        "label": "2. Fuzzy — заменены два слова",
        "query": "The jury returned a verdict of not guilty after four hours of deliberation.",
        "context": None,
    },
    {
        "label": "3. No match — сегмент вне предметной области",
        "query": "The flight controller stabilises the airframe during autonomous flight.",
        "context": None,
    },
]

all_matches: List[Dict[str, Any]] = []

for case in QUERIES_V5:
    matches = calculate_fuzzy_match(
        case["query"],
        my_tm,
        threshold=70.0,
        context_before=case["context"],
    )
    print("=" * 100)
    print(case["label"])
    print(f"Запрос: {case['query']}")

    if not matches:
        print("  -> совпадений выше порога 70 % не найдено")
        continue

    for rank, match in enumerate(matches, start=1):
        print(f"  #{rank} {match['score']:6.2f} %  [{match['match_type']:<12}] tuid={match['tuid']}")
        print(f"      TM src: {match['source_segment']}")
        print(f"      TM tgt: {match['target_segment']}")
        match["query"] = case["query"]
        all_matches.append(match)

matches_df = pd.DataFrame(
    all_matches,
    columns=[
        "query", "tuid", "source_segment", "target_segment",
        "score", "lev_ratio", "token_sort_ratio", "match_type",
    ],
)

smartcat_df = pd.DataFrame({
    "Source (en)": my_glossary["source_term"],
    "Target (ru)": my_glossary["target_term"],
    "Comment": my_glossary["definition"],
})
smartcat_df.to_csv("glossary_smartcat.csv", index=False, encoding="utf-8-sig")

print("=" * 100)
print("Созданы файлы:")
for name in [TBX_FILE, TMX_FILE, "glossary_smartcat.csv"]:
    if os.path.exists(name):
        print(f"  {name:<30} {os.path.getsize(name):>7} байт")

if not matches_df.empty:
    display(matches_df[["query", "tuid", "score", "match_type"]])

Глоссарий: 316 терминов
Память переводов: 62 сегментов
1. ICE — точное совпадение с совпадением контекста
Запрос: The grand jury returned an indictment charging the defendant with fraud.
  #1 100.00 %  [Exact (100%)] tuid=2
      TM src: The grand jury returned an indictment charging the defendant with fraud.
      TM tgt: Большое жюри вынесло обвинительное заключение, обвинив подсудимого в мошенничестве.
2. Fuzzy — заменены два слова
Запрос: The jury returned a verdict of not guilty after four hours of deliberation.
  #1  93.33 %  [Fuzzy       ] tuid=6
      TM src: The jury returned a verdict of not guilty after three hours of deliberation.
      TM tgt: Жюри вынесло вердикт о невиновности после трёх часов обсуждения.
3. No match — сегмент вне предметной области
Запрос: The flight controller stabilises the airframe during autonomous flight.
  -> совпадений выше порога 70 % не найдено
Созданы файлы:
  variant_5.tbx                   172836 байт
  variant_5.tmx                    21049

,query,tuid,score,match_type
0,The grand jury returned an indictment charging the defen...,2,100.00,Exact (100%)
1,The jury returned a verdict of not guilty after four hou...,6,93.33,Fuzzy


## Блок 7. Выводы

Ответьте письменно (3–5 предложений на пункт):

1. Чем концептуально-ориентированная организация TBX отличается от
   сегментно-ориентированной организации TMX?
   
   В стандарте TBX (ISO 30042) одно понятие представлено элементом termEntry, внутри которого для каждого языка создаётся отдельная langSet. Поскольку термин может быть записан как в компактной форме tig, так и в расширенной ntig/termGrp, корректный парсер должен обходить всю langSet, а не опираться на жёстко заданный путь к тегу.

2. Почему `xml:lang` требует особой обработки при парсинге и как вы её решили?
   
   В TMX 1.4b выровненные сегменты хранятся по схеме tu → tuv → seg, а язык варианта задаётся атрибутом xml:lang. При XML-разборе этот атрибут превращается в полное имя с пространством имён, что необходимо учитывать при извлечении языковых вариантов.

3. В каких случаях Levenshtein ratio даёт заниженную оценку сходства и как это компенсирует Token Sort Ratio?

   Расстояние Левенштейна служит базовой метрикой для Fuzzy Match, однако оно сильно зависит от порядка слов. Совместное использование с Token Sort Ratio позволяет находить больше релевантных кандидатов, не снижая точность на почти точных совпадениях.

4. Чем ICE (101 %) отличается от обычного 100 %-совпадения и почему это важно
   для стоимости проекта в Smartcat?

   ICE (In-Context Exact Match) отличается от обычного 100 %-совпадения тем, что учитывает соседние сегменты. Именно ICE даёт возможность подставить готовый перевод без дополнительной проверки редактором.

5. Какие проблемы возникли при автоматической подстановке терминов и как вы их
   решили?
   
   При автоматической разметке терминов необходимо сначала обрабатывать более длинные термины и защищать уже размеченные фрагменты. Иначе вложенные подстановки могут разрушить многословные термины и исказить результат.

**Ваши выводы:**

В результате выполнения работы я научился программно обрабатывать стандарты обмена терминологическими данными (TBX) и память переводов (TMX), реализовывать поиск нечётких совпадений (Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в CAT-систему Smartcat.

---

### Что сдать на проверку

1. Ссылка на Colab-ноутбук `lab_01_Фамилия.ipynb` (доступ «для всех по ссылке»).
2. Файлы `variant_NN.tbx`, `variant_NN.tmx`, `glossary_smartcat.csv`.
3. Отчёт с выводами и скриншотом импортированного глоссария в Smartcat.